# Ecommerce Customer Churn Analysis

Every cell below is a raw SQL statement run against an in-memory SQLite copy of `Data.csv`; Python only loads the CSV and executes the statements.

In [63]:
import sqlite3
import pandas as pd

con = sqlite3.connect(":memory:")

# convert_dtypes keeps whole-number columns as INTEGER, so SQLite returns integer sums and means.
pd.read_csv("Data.csv").convert_dtypes().to_sql("ecommercechurn", con, index=False)

def sql(query):
    """Run one statement: a DataFrame for SELECTs, the affected row count otherwise."""
    if query.lstrip().upper().startswith("SELECT"):
        return pd.read_sql_query(query, con)
    return f"{con.execute(query).rowcount} rows affected"

sql("SELECT COUNT(*) AS rows FROM ecommercechurn")

,rows
0,5630


## Data cleaning



### 1-3. Customer count, duplicates, nulls

*Note: wrap the cleaning mutations below in a single `BEGIN...COMMIT` transaction. SQLite auto-commits; these run as implicit transactions, each statement is atomic.*

In [64]:
sql("""
-- Single UPDATE with COALESCE replaces 7 identical UPDATEs. ceiling = per-column strategies if types diverge.
UPDATE ecommercechurn SET
  HourSpendOnApp = COALESCE(HourSpendOnApp, (SELECT CAST(AVG(HourSpendOnApp) AS INT) FROM ecommercechurn)),
  Tenure = COALESCE(Tenure, (SELECT CAST(AVG(Tenure) AS INT) FROM ecommercechurn)),
  OrderAmountHikeFromLastYear = COALESCE(OrderAmountHikeFromLastYear, (SELECT CAST(AVG(OrderAmountHikeFromLastYear) AS INT) FROM ecommercechurn)),
  WarehouseToHome = COALESCE(WarehouseToHome, (SELECT CAST(AVG(WarehouseToHome) AS INT) FROM ecommercechurn)),
  CouponUsed = COALESCE(CouponUsed, (SELECT CAST(AVG(CouponUsed) AS INT) FROM ecommercechurn)),
  OrderCount = COALESCE(OrderCount, (SELECT CAST(AVG(OrderCount) AS INT) FROM ecommercechurn)),
  DaySinceLastOrder = COALESCE(DaySinceLastOrder, (SELECT CAST(AVG(DaySinceLastOrder) AS INT) FROM ecommercechurn));
""")

sql("""
SELECT COUNT(*) AS RemainingNulls
FROM ecommercechurn
WHERE HourSpendOnApp IS NULL
   OR Tenure IS NULL
   OR OrderAmountHikeFromLastYear IS NULL
   OR WarehouseToHome IS NULL
   OR CouponUsed IS NULL
   OR OrderCount IS NULL
   OR DaySinceLastOrder IS NULL
""")

,RemainingNulls
0,0


In [65]:
print('Step 3.1: Nulls filled successfully.')

Step 3.1: Nulls filled successfully.


In [66]:
sql("""
SELECT CustomerID, COUNT(CustomerID) AS Count
FROM ecommercechurn
GROUP BY CustomerID
HAVING COUNT(CustomerID) > 1
""")

,CustomerID,Count


In [67]:
sql("""
SELECT 'Tenure' AS ColumnName, COUNT(*) AS NullCount
FROM ecommercechurn
WHERE Tenure IS NULL

UNION

SELECT 'WarehouseToHome', COUNT(*)
FROM ecommercechurn
WHERE WarehouseToHome IS NULL

UNION

SELECT 'HourSpendOnApp', COUNT(*)
FROM ecommercechurn
WHERE HourSpendOnApp IS NULL

UNION

SELECT 'OrderAmountHikeFromLastYear', COUNT(*)
FROM ecommercechurn
WHERE OrderAmountHikeFromLastYear IS NULL

UNION

SELECT 'CouponUsed', COUNT(*)
FROM ecommercechurn
WHERE CouponUsed IS NULL

UNION

SELECT 'OrderCount', COUNT(*)
FROM ecommercechurn
WHERE OrderCount IS NULL

UNION

SELECT 'DaySinceLastOrder', COUNT(*)
FROM ecommercechurn
WHERE DaySinceLastOrder IS NULL
""")

,ColumnName,NullCount
0,CouponUsed,0
1,DaySinceLastOrder,0
2,HourSpendOnApp,0
3,OrderAmountHikeFromLastYear,0
4,OrderCount,0
5,Tenure,0
6,WarehouseToHome,0


### 3.1. Fill nulls with the column mean

In [68]:
sql("""
UPDATE ecommercechurn SET HourSpendOnApp = (SELECT CAST(AVG(HourSpendOnApp) AS INT) FROM ecommercechurn)
WHERE HourSpendOnApp IS NULL
""")

sql("""
UPDATE ecommercechurn SET Tenure = (SELECT CAST(AVG(Tenure) AS INT) FROM ecommercechurn)
WHERE Tenure IS NULL
""")

sql("""
UPDATE ecommercechurn SET OrderAmountHikeFromLastYear = (SELECT CAST(AVG(OrderAmountHikeFromLastYear) AS INT) FROM ecommercechurn)
WHERE OrderAmountHikeFromLastYear IS NULL
""")

sql("""
UPDATE ecommercechurn SET WarehouseToHome = (SELECT CAST(AVG(WarehouseToHome) AS INT) FROM ecommercechurn)
WHERE WarehouseToHome IS NULL
""")

sql("""
UPDATE ecommercechurn SET CouponUsed = (SELECT CAST(AVG(CouponUsed) AS INT) FROM ecommercechurn)
WHERE CouponUsed IS NULL
""")

sql("""
UPDATE ecommercechurn SET OrderCount = (SELECT CAST(AVG(OrderCount) AS INT) FROM ecommercechurn)
WHERE OrderCount IS NULL
""")

sql("""
UPDATE ecommercechurn SET DaySinceLastOrder = (SELECT CAST(AVG(DaySinceLastOrder) AS INT) FROM ecommercechurn)
WHERE DaySinceLastOrder IS NULL
""")

sql("""
SELECT COUNT(*) AS RemainingNulls
FROM ecommercechurn
WHERE HourSpendOnApp IS NULL
   OR Tenure IS NULL
   OR OrderAmountHikeFromLastYear IS NULL
   OR WarehouseToHome IS NULL
   OR CouponUsed IS NULL
   OR OrderCount IS NULL
   OR DaySinceLastOrder IS NULL
""")

,RemainingNulls
0,0


### 4-5. Readable status columns (replaces `Churn` / `Complain` 0-1 values)

In [69]:
sql("ALTER TABLE ecommercechurn ADD CustomerStatus NVARCHAR(50)")

sql("""
UPDATE ecommercechurn
SET CustomerStatus = CASE
    WHEN Churn = 1 THEN 'Churned'
    WHEN Churn = 0 THEN 'Stayed'
END
""")

sql("ALTER TABLE ecommercechurn ADD ComplainRecieved NVARCHAR(10)")

sql("""
UPDATE ecommercechurn
SET ComplainRecieved = CASE
    WHEN Complain = 1 THEN 'Yes'
    WHEN Complain = 0 THEN 'No'
END
""")

sql("""
SELECT CustomerStatus, ComplainRecieved, COUNT(*) AS Customers
FROM ecommercechurn
GROUP BY CustomerStatus, ComplainRecieved
""")

,CustomerStatus,ComplainRecieved,Customers
0,Churned,No,440
1,Churned,Yes,508
2,Stayed,No,3586
3,Stayed,Yes,1096


### 6. Check and normalise inconsistent values

In [70]:
sql("""
SELECT 'PreferredLoginDevice' AS FieldName, PreferredLoginDevice AS Value
FROM ecommercechurn
GROUP BY PreferredLoginDevice

UNION ALL

SELECT 'PreferedOrderCat', PreferedOrderCat
FROM ecommercechurn
GROUP BY PreferedOrderCat
UNION ALL

SELECT 'PreferredPaymentMode', PreferredPaymentMode
FROM ecommercechurn
GROUP BY PreferredPaymentMode
""")

,FieldName,Value
0,PreferredLoginDevice,Computer
1,PreferredLoginDevice,Mobile Phone
2,PreferredLoginDevice,Phone
3,PreferedOrderCat,Fashion
4,PreferedOrderCat,Grocery
5,PreferedOrderCat,Laptop & Accessory
6,PreferedOrderCat,Mobile
7,PreferedOrderCat,Mobile Phone
8,PreferedOrderCat,Others
9,PreferredPaymentMode,CC


In [71]:
sql("""
UPDATE ecommercechurn SET PreferredLoginDevice = 'Phone'
WHERE PreferredLoginDevice = 'mobile phone' COLLATE NOCASE
""")

sql("""
UPDATE ecommercechurn SET PreferedOrderCat = 'Mobile Phone'
WHERE PreferedOrderCat = 'Mobile'
""")

sql("""
UPDATE ecommercechurn SET PreferredPaymentMode = 'Cash on Delivery'
WHERE PreferredPaymentMode = 'COD'
""")

sql("UPDATE ecommercechurn SET WarehouseToHome = CAST('27' AS INT) WHERE WarehouseToHome = '127'")
sql("UPDATE ecommercechurn SET WarehouseToHome = CAST('26' AS INT) WHERE WarehouseToHome = '126'")

sql("SELECT MAX(WarehouseToHome) AS MaxWarehouseToHome FROM ecommercechurn")

,MaxWarehouseToHome
0,36


### Derived ranges (used by questions 4, 6 and 17)

In [72]:
sql("ALTER TABLE ecommercechurn ADD WarehouseToHomeRange NVARCHAR(50)")

sql("""
UPDATE ecommercechurn
SET WarehouseToHomeRange = CASE
    WHEN WarehouseToHome <= 10 THEN 'Very close distance'
    WHEN WarehouseToHome > 10 AND WarehouseToHome <= 20 THEN 'Close distance'
    WHEN WarehouseToHome > 20 AND WarehouseToHome <= 30 THEN 'Moderate distance'
    WHEN WarehouseToHome > 30 THEN 'Far distance'
END
""")

sql("ALTER TABLE ecommercechurn ADD TenureRange NVARCHAR(50)")

sql("""
UPDATE ecommercechurn
SET TenureRange = CASE
    WHEN Tenure <= 6 THEN '6 Months'
    WHEN Tenure > 6 AND Tenure <= 12 THEN '1 Year'
    WHEN Tenure > 12 AND Tenure <= 24 THEN '2 Years'
    WHEN Tenure > 24 THEN 'more than 2 years'
END
""")



sql("ALTER TABLE ecommercechurn ADD CashbackAmountRange NVARCHAR(50)")

sql("""
UPDATE ecommercechurn
SET CashbackAmountRange = CASE
    WHEN CashbackAmount <= 100 THEN 'Low Cashback Amount'
    WHEN CashbackAmount > 100 AND CashbackAmount <= 200 THEN 'Moderate Cashback Amount'
    WHEN CashbackAmount > 200 AND CashbackAmount <= 300 THEN 'High Cashback Amount'
    WHEN CashbackAmount > 300 THEN 'Very High Cashback Amount'
END
""")

sql("""
SELECT WarehouseToHomeRange, TenureRange, CashbackAmountRange
FROM ecommercechurn
LIMIT 5
""")

,WarehouseToHomeRange,TenureRange,CashbackAmountRange
0,Very close distance,6 Months,Moderate Cashback Amount
1,Very close distance,1 Year,Moderate Cashback Amount
2,Moderate distance,1 Year,Moderate Cashback Amount
3,Close distance,6 Months,Moderate Cashback Amount
4,Close distance,6 Months,Moderate Cashback Amount


## Analysis (requires cleaned table above)



### 1. What is the overall customer churn rate?

> The churn rate is 16.84%.

In [73]:
sql("""
SELECT TotalNumberofCustomers,
TotalNumberofChurnedCustomers,
ROUND(TotalNumberofChurnedCustomers * 1.0 / TotalNumberofCustomers * 100, 2) AS ChurnRate
FROM
(SELECT COUNT(*) AS TotalNumberofCustomers
FROM ecommercechurn) AS Total,
(SELECT COUNT(*) AS TotalNumberofChurnedCustomers
FROM ecommercechurn
WHERE CustomerStatus = 'Churned' COLLATE NOCASE) AS Churned
""")

,TotalNumberofCustomers,TotalNumberofChurnedCustomers,ChurnRate
0,5630,948,16.84


### 2. How does the churn rate vary based on the preferred login device?

> Computer has the highest churn rate at 19.83%, then phone with 15.62%.

In [74]:
sql("""
SELECT PreferredLoginDevice,
COUNT(*) AS TotalCustomers,
SUM(Churn) AS ChurnedCustomers,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS ChurnRate
FROM ecommercechurn
GROUP BY PreferredLoginDevice
""")

,PreferredLoginDevice,TotalCustomers,ChurnedCustomers,ChurnRate
0,Computer,1634,324,19.83
1,Phone,3996,624,15.62


### 3. What is the distribution of customers across different city tiers?

> City tier 3 has the highest churn rate, then tier 2, then tier 1 with the least.

In [75]:
sql("""
SELECT CityTier,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS ChurnedCustomers,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS ChurnRate
FROM ecommercechurn
GROUP BY CityTier
ORDER BY ChurnRate DESC
""")

,CityTier,TotalCustomer,ChurnedCustomers,ChurnRate
0,3,1722,368,21.37
1,2,242,48,19.83
2,1,3666,532,14.51


### 4. Is there any correlation between the warehouse-to-home distance and customer churn?

> The churn rate increases as the warehouse-to-home distance increases.

In [76]:
sql("""
SELECT WarehouseToHomeRange,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY WarehouseToHomeRange
ORDER BY Churnrate DESC""")

,WarehouseToHomeRange,TotalCustomer,CustomerChurn,Churnrate
0,Far distance,469,98,20.90
1,Moderate distance,874,176,20.14
2,Close distance,2318,408,17.60
3,Very close distance,1969,266,13.51


### 5. Which is the most preferred payment mode among churned customers?

> Cash on Delivery has the highest churn rate (24.90%).
#### Note on Question 5 semantics
> This query ranks payment modes by churn rate among customers grouped by that mode.
> It does NOT answer 'which payment mode is most used by churned customers.'
> For that, filter WHERE CustomerStatus='Churned' before grouping.

In [77]:
sql("""
SELECT PreferredPaymentMode,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY PreferredPaymentMode
ORDER BY Churnrate DESC
""")

,PreferredPaymentMode,TotalCustomer,CustomerChurn,Churnrate
0,Cash on Delivery,514,128,24.90
1,E wallet,614,140,22.80
2,CC,273,59,21.61
3,UPI,414,72,17.39
4,Debit Card,2314,356,15.38
5,Credit Card,1501,193,12.86


### 6. What is the typical tenure for churned customers?

> Most customers churned within a 6 month tenure period.

In [78]:
sql("""
SELECT TenureRange,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY TenureRange
ORDER BY Churnrate DESC
""")

,TenureRange,TotalCustomer,CustomerChurn,Churnrate
0,6 Months,2150,697,32.42
1,1 Year,1584,156,9.85
2,2 Years,1467,95,6.48
3,more than 2 years,429,0,0.00


### 7. Is there any difference in churn rate between male and female customers?

> More men churned compared to women.

In [79]:
sql("""
SELECT Gender,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY Gender
ORDER BY Churnrate DESC""")

,Gender,TotalCustomer,CustomerChurn,Churnrate
0,Male,3384,600,17.73
1,Female,2246,348,15.49


### 8. How does the average time spent on the app differ for churned and non-churned customers?

> There is no difference between the average time spent on the app for churned and non-churned customers.

In [80]:
sql("""
SELECT CustomerStatus, AVG(HourSpendOnApp) AS AverageHourSpentonApp
FROM ecommercechurn
GROUP BY CustomerStatus
""")

,CustomerStatus,AverageHourSpentonApp
0,Churned,2.902954
1,Stayed,2.886587


### 9. Does the number of registered devices impact the likelihood of churn?

> As the number of registered devices increases the churn rate increases.

In [81]:
sql("""
SELECT NumberOfDeviceRegistered,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY NumberOfDeviceRegistered
ORDER BY Churnrate DESC
""")

,NumberOfDeviceRegistered,TotalCustomer,CustomerChurn,Churnrate
0,6,162,56,34.57
1,5,881,198,22.47
2,4,2377,392,16.49
3,3,1699,254,14.95
4,2,276,26,9.42
5,1,235,22,9.36


### 10. Which order category is most preferred among churned customers?

> Mobile Phone has the highest churn rate and Grocery the least.

In [82]:
sql("""
SELECT PreferedOrderCat,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY PreferedOrderCat
ORDER BY Churnrate DESC
""")

,PreferedOrderCat,TotalCustomer,CustomerChurn,Churnrate
0,Mobile Phone,2080,570,27.40
1,Fashion,826,128,15.50
2,Laptop & Accessory,2050,210,10.24
3,Others,264,20,7.58
4,Grocery,410,20,4.88


### 11. Is there any relationship between customer satisfaction scores and churn?

> Satisfaction score 5 has the highest churn rate, score 1 the least.

In [83]:
sql("""
SELECT SatisfactionScore,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY SatisfactionScore
ORDER BY Churnrate DESC
""")

,SatisfactionScore,TotalCustomer,CustomerChurn,Churnrate
0,5,1108,264,23.83
1,3,1698,292,17.20
2,4,1074,184,17.13
3,2,586,74,12.63
4,1,1164,134,11.51


### 12. Does the marital status of customers influence churn behavior?

> Single customers have the highest churn rate, married customers the least.

In [84]:
sql("""
SELECT MaritalStatus,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY MaritalStatus
ORDER BY Churnrate DESC
""")

,MaritalStatus,TotalCustomer,CustomerChurn,Churnrate
0,Single,1796,480,26.73
1,Divorced,848,124,14.62
2,Married,2986,344,11.52


### 13. How many addresses do churned customers have on average?

> Churned customers have about 4.5 addresses on average. The original script filtered `stayed` here, which contradicted the question and its stated answer.

In [85]:
sql("""
SELECT AVG(NumberOfAddress) AS Averagenumofchurnedcustomeraddress
FROM ecommercechurn
WHERE CustomerStatus = 'Churned' COLLATE NOCASE
""")

,Averagenumofchurnedcustomeraddress
0,4.466245


### 14. Does customer complaints influence churned behavior?

> Customers with complaints had the highest churn rate.

In [86]:
sql("""
SELECT ComplainRecieved,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY ComplainRecieved
ORDER BY Churnrate DESC
""")

,ComplainRecieved,TotalCustomer,CustomerChurn,Churnrate
0,Yes,1604,508,31.67
1,No,4026,440,10.93


### 15. How does the usage of coupons differ between churned and non-churned customers?

> Churned customers used fewer coupons than non-churned customers.

In [87]:
sql("""
SELECT CustomerStatus, SUM(CouponUsed) AS SumofCouponUsed
FROM ecommercechurn
GROUP BY CustomerStatus
""")

,CustomerStatus,SumofCouponUsed
0,Churned,1622
1,Stayed,8044


### 16. What is the average number of days since the last order for churned customers?

> The average number of days since the last order for churned customers is 3.

In [88]:
sql("""
SELECT AVG(DaySinceLastOrder) AS AverageNumofDaysSinceLastOrder
FROM ecommercechurn
WHERE CustomerStatus = 'Churned' COLLATE NOCASE
""")

,AverageNumofDaysSinceLastOrder
0,3.279536


### 17. Is there any correlation between cashback amount and churn rate?

> Moderate cashback (100-200) has the highest churn rate, followed by high, very high and then low.

In [89]:
sql("""
SELECT CashbackAmountRange,
COUNT(*) AS TotalCustomer,
SUM(Churn) AS CustomerChurn,
ROUND(SUM(Churn) * 1.0 / COUNT(*) * 100, 2) AS Churnrate
FROM ecommercechurn
GROUP BY CashbackAmountRange
ORDER BY Churnrate DESC
""")

,CashbackAmountRange,TotalCustomer,CustomerChurn,Churnrate
0,Moderate Cashback Amount,4305,814,18.91
1,High Cashback Amount,1157,124,10.72
2,Very High Cashback Amount,156,10,6.41
3,Low Cashback Amount,12,0,0.00
